# Spark Performance — Higher-Order Functions vs Explode (Task 2.3)

Three analytics on `order_payments_nested` — each solved **twice**: higher-order functions (`filter`, `transform`, `aggregate`) vs **explode** / flattened table. Compare `explain()` and timing.

In [ ]:
import sys
import importlib

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src.spark_performance.higher_order_payments as ho_module
importlib.reload(ho_module)

from pyspark.sql import functions as F
from src.spark_performance.execution_plans import capture_explain, benchmark_df
from src.spark_performance.higher_order_payments import (
    NestedPaymentsRefs,
    ho_orders_high_credit_card,
    explode_orders_high_credit_card,
    ho_total_credit_card_value,
    explode_total_credit_card_value,
    ho_max_non_boleto_installments,
    explode_max_non_boleto_installments,
    compare_approaches,
    OBSERVATIONS,
)
from src.ingestion.idempotent_loader import save_run_report_to_volume

refs = NestedPaymentsRefs()
nested = spark.table(refs.nested_table)
flattened = spark.table(refs.flattened_table)
print("Loaded from:", ho_module.__file__)
print("Nested rows:", nested.count(), "| Flattened rows:", flattened.count())

## Problem 1 — Filter: orders with credit_card payment > R$100

In [ ]:
p1_ho = ho_orders_high_credit_card(nested)
p1_ex = explode_orders_high_credit_card(flattened)
p1 = compare_approaches(p1_ho, p1_ex, "filter_high_credit_card")
print("HO explain:\n", capture_explain(p1_ho))
print("Explode explain:\n", capture_explain(p1_ex))
display(p1_ho.limit(5))
print(p1)

## Problem 2 — Aggregate: total credit_card value per order

In [ ]:
p2_ho = ho_total_credit_card_value(nested)
p2_ex = explode_total_credit_card_value(flattened)
p2 = compare_approaches(p2_ho, p2_ex, "aggregate_credit_card_total")
print("HO explain:\n", capture_explain(p2_ho))
print("Explode explain:\n", capture_explain(p2_ex))
display(p2_ho.orderBy(F.col("credit_card_total").desc()).limit(5))
print(p2)

## Problem 3 — Transform + aggregate: max installments (non-boleto) per order

In [ ]:
p3_ho = ho_max_non_boleto_installments(nested)
p3_ex = explode_max_non_boleto_installments(flattened)
p3 = compare_approaches(p3_ho, p3_ex, "transform_max_non_boleto_installments")
print("HO explain:\n", capture_explain(p3_ho))
print("Explode explain:\n", capture_explain(p3_ex))
display(p3_ho.orderBy(F.col("max_non_boleto_installments").desc()).limit(5))
print(p3)

In [ ]:
import json

report = {
    "task": "2.3_higher_order_vs_explode",
    "nested_table": refs.nested_table,
    "flattened_table": refs.flattened_table,
    "problems": [p1, p2, p3],
    "observations": OBSERVATIONS,
}
print(json.dumps(report, indent=2, default=str))
print("\n--- Observations ---")
for line in OBSERVATIONS:
    print("-", line)
try:
    save_run_report_to_volume(
        report,
        dbutils,
        "/Volumes/globalmart/metadata/run_reports/m02_task23_higher_order.json",
    )
except Exception as exc:
    print("Volume save skipped:", exc)